# Zapas Topic 3 — Monthly store/product data pipeline

Parts A–C; Pathway F. All categories, stores, raw rows and attributes retained. No model training. The saved outputs show a local execution; hosted Colab execution has not been independently performed.

Actual local inputs: Topic 3_Lab.docx and archive (13).zip; equivalence to the suffixed chat attachments is unverified. Requirements: FR1/FR2, restocking part of FR4, RAI1, NFR4. Historical Week 2 sentiment scope is superseded by the current instruction.

## Optional dependency setup
Run in a fresh Colab runtime for pinned dependencies, then restart the runtime if requested. The tested Python version is 3.11.5. This setup is intentionally disabled for ordinary reruns.

In [1]:
INSTALL_PINNED = False
if INSTALL_PINNED:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas==2.2.3', 'numpy==1.26.4'])

## A2 — Cell 1: Workspace
Colab uses the lab standard path. Local automation may set ZAPAS_BASE_DIR.

In [2]:
from pathlib import Path
import os
BASE_DIR = Path(os.environ.get('ZAPAS_BASE_DIR', '/content/week3_project' if Path('/content').exists() else 'week3_project')).resolve()
RAW_DIR = BASE_DIR / 'data/raw'
PROCESSED_DIR = BASE_DIR / 'data/processed'
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW = RAW_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR
print('Workspace:', BASE_DIR)

Workspace: C:\Users\xhazi\Documents\Codex\2026-11-03\referenced-chatgpt-conversation-this-is-an\work\notebook_run


## Cell 2: Upload the project CSV or ZIP
Skip upload if data are already in data/raw. Only the exact named CSV is extracted; the original archive stays unchanged.

In [3]:
if not (RAW_DIR / 'retail_store_inventory.csv').exists() and not list(RAW_DIR.glob('*.zip')):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place the original CSV or ZIP in ' + str(RAW_DIR))
    uploaded = files.upload()
    for filename, payload in uploaded.items():
        (RAW_DIR / Path(filename).name).write_bytes(payload)
print([(path.name, path.stat().st_size) for path in RAW_DIR.iterdir() if path.is_file()])

[('archive (13).zip', 1588139)]


## Cells 4, 4A and 7: Shared source, transformation and lineage utilities
The lab helpers are consolidated into reusable functions. run_pipeline registers actual source/output relationships and saves manifests. Part B and Part C invoke the same transformation. Invalid dates and ambiguous duplicate daily keys stop processing without deleting data.

In [4]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import sys
import zipfile

import numpy as np
import pandas as pd


VERSION = 'zapas-week3-v1.0'
KEYS = ['Store ID', 'Product ID']
REQUIRED = ['Date', 'Store ID', 'Product ID', 'Category', 'Region',
            'Inventory Level', 'Units Sold', 'Units Ordered', 'Demand Forecast',
            'Price', 'Discount', 'Weather Condition', 'Holiday/Promotion',
            'Competitor Pricing', 'Seasonality']
FEATURES = ['lag_1', 'lag_2', 'lag_3', 'lag_12', 'rolling_mean_3',
            'rolling_mean_6', 'sales_trend', 'month_number']
CONFIG = {'pathway': 'F', 'time_column': 'Date', 'frequency': 'MS',
          'keys': KEYS, 'invalid_key_policy': 'stop', 'duplicate_key_policy': 'stop',
          'missing_sales_policy': 'preserve_and_flag', 'category_filter': None,
          'forecast_horizon_months': 1, 'minimum_history_months': 3}


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False,
                                    allow_nan=False) + '\n', encoding='utf-8')


def select_source(raw_dir):
    raw_dir = Path(raw_dir)
    source = raw_dir / 'retail_store_inventory.csv'
    if source.is_file():
        return source
    candidates = []
    for archive in sorted(raw_dir.glob('*.zip')):
        with zipfile.ZipFile(archive) as bundle:
            candidates.extend((archive, item.filename) for item in bundle.infolist()
                              if Path(item.filename).name == source.name)
    if len(candidates) != 1:
        raise ValueError('Upload exactly one retail_store_inventory.csv or ZIP containing it into data/raw.')
    archive, member = candidates[0]
    with zipfile.ZipFile(archive) as bundle:
        source.write_bytes(bundle.read(member))
    return source


def inspect_source(source):
    frame = pd.read_csv(source)
    if frame.empty or not set(REQUIRED).issubset(frame.columns):
        raise ValueError('Empty source or missing required columns.')
    dates = pd.to_datetime(frame['Date'], format='%Y-%m-%d', errors='coerce')
    if dates.isna().any() or frame[KEYS].isna().any().any():
        raise ValueError('Invalid date or missing series identifier; source preserved, pipeline stopped.')
    if frame.duplicated(['Date'] + KEYS).any():
        raise ValueError('Duplicate store/product/day key; resolve source ambiguity before aggregation.')
    sales = pd.to_numeric(frame['Units Sold'], errors='coerce')
    if (frame['Units Sold'].notna() & sales.isna()).any() or (sales.dropna() < 0).any() or np.isinf(sales.dropna()).any():
        raise ValueError('Units Sold must be finite nonnegative numbers or explicit missing values.')
    if not sales.dropna().mod(1).eq(0).all():
        raise ValueError('Units Sold must contain whole-unit counts.')
    frame['Date'] = dates
    frame['Units Sold'] = sales
    report = {'rows': len(frame), 'columns': list(frame.columns),
              'dtypes': {column: str(dtype) for column, dtype in frame.dtypes.items()},
              'missing_by_column': frame.isna().sum().astype(int).to_dict(),
              'date_start': str(dates.min().date()), 'date_end': str(dates.max().date()),
              'stores': int(frame['Store ID'].nunique()),
              'products': int(frame['Product ID'].nunique()),
              'series': len(frame[KEYS].drop_duplicates()),
              'categories': sorted(frame.Category.dropna().unique().tolist()),
              'duplicate_daily_keys': 0, 'invalid_timestamps': 0,
              'products_with_multiple_categories': int((frame.groupby('Product ID').Category.nunique() > 1).sum()),
              'stores_with_multiple_regions': int((frame.groupby('Store ID').Region.nunique() > 1).sum())}
    return frame, report


def prepare(frame):
    daily = frame.sort_values(['Date'] + KEYS, kind='stable').reset_index(drop=True)
    observed = daily.assign(month=daily.Date.dt.to_period('M').dt.to_timestamp())
    grouped = observed.groupby(KEYS + ['month'], sort=True)
    monthly = grouped.agg(observed_days=('Date', 'nunique'),
                          nonmissing_sales_days=('Units Sold', 'count'),
                          units_sold=('Units Sold', lambda values: values.sum(min_count=1)),
                          inventory_last_observed=('Inventory Level', lambda values: values.iloc[-1]),
                          last_observed_date=('Date', 'max'),
                          category_count=('Category', 'nunique'),
                          region_count=('Region', 'nunique')).reset_index()
    calendar = pd.date_range(observed.month.min(), observed.month.max(), freq='MS')
    grid = observed[KEYS].drop_duplicates().merge(pd.DataFrame({'month': calendar}), how='cross')
    monthly = grid.merge(monthly, on=KEYS + ['month'], how='left', validate='one_to_one')
    monthly = monthly.sort_values(KEYS + ['month']).reset_index(drop=True)
    monthly['observed_days'] = monthly.observed_days.fillna(0).astype(int)
    monthly['nonmissing_sales_days'] = monthly.nonmissing_sales_days.fillna(0).astype(int)
    monthly['expected_days'] = monthly.month.dt.days_in_month
    monthly['complete_month'] = ((monthly.observed_days == monthly.expected_days) &
                                 (monthly.nonmissing_sales_days == monthly.expected_days))
    monthly['target_units_sold'] = monthly.units_sold.where(monthly.complete_month)
    monthly['month_number'] = monthly.month.dt.month
    series = monthly.groupby(KEYS, sort=False)['target_units_sold']
    for lag in [1, 2, 3, 12]:
        monthly[f'lag_{lag}'] = series.shift(lag)
    for window in [3, 6]:
        monthly[f'rolling_mean_{window}'] = series.transform(
            lambda values: values.shift(1).rolling(window, min_periods=window).mean())
    monthly['sales_trend'] = monthly.lag_1 - monthly.lag_2
    monthly['history_ready'] = monthly[['lag_1', 'lag_2', 'lag_3']].notna().all(axis=1)
    monthly['eligible_for_training'] = monthly.complete_month & monthly.history_ready
    monthly['split'] = 'ineligible'
    eligible_months = sorted(monthly.loc[monthly.eligible_for_training, 'month'].unique())
    if len(eligible_months) >= 7:
        train_end, validation_end = eligible_months[-7], eligible_months[-4]
        eligible = monthly.eligible_for_training
        monthly.loc[eligible & (monthly.month <= train_end), 'split'] = 'train'
        monthly.loc[eligible & (monthly.month > train_end) & (monthly.month <= validation_end), 'split'] = 'validation'
        monthly.loc[eligible & (monthly.month > validation_end), 'split'] = 'test'
    else:
        monthly.loc[monthly.eligible_for_training, 'split'] = 'train'
    return daily, monthly


def export_and_validate(source, destination, base):
    destination, base = Path(destination), Path(base)
    frame, inspection = inspect_source(source)
    daily, monthly = prepare(frame)
    destination.mkdir(parents=True, exist_ok=True)
    outputs = []
    for filename, result in [('prepared_timeseries.csv', daily), ('monthly_sales_features.csv', monthly)]:
        path = destination / filename
        result.to_csv(path, index=False, date_format='%Y-%m-%d', lineterminator='\n')
        loaded = pd.read_csv(path)
        expected = result.copy()
        for column in expected.select_dtypes(include=['datetime']).columns:
            expected[column] = expected[column].dt.strftime('%Y-%m-%d')
        pd.testing.assert_frame_equal(loaded, expected, check_dtype=False, check_exact=False)
        outputs.append(path)
    if len(daily) != len(frame) or list(daily.columns) != list(frame.columns):
        raise AssertionError('Daily export lost rows or columns.')
    if not daily.Date.is_monotonic_increasing or monthly.duplicated(KEYS + ['month']).any():
        raise AssertionError('Invalid ordering or monthly uniqueness.')
    if not np.isclose(daily['Units Sold'].sum(), monthly.units_sold.sum()):
        raise AssertionError('Monthly totals do not reconcile to raw sales.')
    if set(map(tuple, daily[KEYS].to_numpy())) != set(map(tuple, monthly[KEYS].to_numpy())):
        raise AssertionError('Store/product series were lost.')
    if monthly.loc[~monthly.complete_month, 'target_units_sold'].notna().any():
        raise AssertionError('Incomplete month exposed as a training target.')
    inspection.update({'monthly_rows': len(monthly),
                       'complete_month_rows': int(monthly.complete_month.sum()),
                       'incomplete_month_rows': int((~monthly.complete_month).sum()),
                       'eligible_rows': int(monthly.eligible_for_training.sum()),
                       'split_counts': monthly.split.value_counts().to_dict(),
                       'total_units_sold': int(daily['Units Sold'].sum()),
                       'raw_preserved': True, 'validation': 'PASS'})
    quality = destination / 'validation_report.json'
    write_json(quality, inspection)
    outputs.append(quality)
    return outputs, inspection


def run_pipeline(base, stage='pipeline_v1/timeseries'):
    base = Path(base).resolve()
    source = select_source(base / 'data/raw')
    before = sha256_file(source)
    destination = base / 'data' / stage
    outputs, inspection = export_and_validate(source, destination, base)
    if before != sha256_file(source):
        raise AssertionError('Raw source changed.')
    records = [{'source_id': 'S01', 'source_filename': 'retail_store_inventory.csv',
                'source_sha256': before, 'source_size_bytes': source.stat().st_size,
                'prepared_output': path.relative_to(base).as_posix(),
                'prepared_sha256': sha256_file(path), 'size_bytes': path.stat().st_size,
                'processing_note': CONFIG, 'source_version': 'Local file fingerprint',
                'access_rights': 'Dataset licence not verified; confirm before public redistribution'}
               for path in outputs]
    manifest = {'pipeline_version': VERSION, 'generated_at_utc': datetime.now(timezone.utc).isoformat(),
                'runtime': {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__},
                'records': records, 'inspection': inspection, 'model_feature_allowlist': FEATURES,
                'source_output_mapping': [{'source': 'data/raw/retail_store_inventory.csv',
                                           'output': record['prepared_output']} for record in records]}
    manifest_path = base / 'data/week3_manifest.json' if stage == 'processed' else destination / 'pipeline_manifest.json'
    write_json(manifest_path, manifest)
    print(f'SUCCESS: {stage}: {inspection["rows"]} daily rows; {inspection["monthly_rows"]} monthly rows')
    return manifest


def verify_repeatability(first, second):
    for key in ['records', 'inspection', 'source_output_mapping', 'model_feature_allowlist']:
        if first[key] != second[key]:
            raise AssertionError(f'Repeatability failed: {key}')
    return {'status': 'PASS', 'checks': ['source hashes and sizes', 'output hashes and sizes',
                                       'counts', 'source-output mappings', 'feature allowlist'],
            'scope': 'Same-runtime repeatability; execution timestamps intentionally excluded'}




## Cell 3 and A3: Verify and inspect two subsets

In [5]:
source = select_source(RAW_DIR)
frame, inspection = inspect_source(source)
print(source.name, source.stat().st_size, sha256_file(source))
print(json.dumps(inspection, indent=2))
for product in ['P0001', 'P0002']:
    print(product)
    display(frame.loc[frame['Product ID'].eq(product)].head())

retail_store_inventory.csv 6191463 c7b6019887d85e6bb311a76da72560463e6bf269bfadbe4b326fbff3b1000468
{
  "rows": 73100,
  "columns": [
    "Date",
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Demand Forecast",
    "Price",
    "Discount",
    "Weather Condition",
    "Holiday/Promotion",
    "Competitor Pricing",
    "Seasonality"
  ],
  "dtypes": {
    "Date": "datetime64[ns]",
    "Store ID": "object",
    "Product ID": "object",
    "Category": "object",
    "Region": "object",
    "Inventory Level": "int64",
    "Units Sold": "int64",
    "Units Ordered": "int64",
    "Demand Forecast": "float64",
    "Price": "float64",
    "Discount": "int64",
    "Weather Condition": "object",
    "Holiday/Promotion": "int64",
    "Competitor Pricing": "float64",
    "Seasonality": "object"
  },
  "missing_by_column": {
    "Date": 0,
    "Store ID": 0,
    "Product ID": 0,
    "Category": 0,
    "Region": 0,
  

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
20,2022-01-01,S002,P0001,Groceries,South,343,104,144,112.55,32.80,20,Sunny,1,30.78,Spring
40,2022-01-01,S003,P0001,Toys,South,241,147,180,161.09,67.21,5,Sunny,1,67.39,Summer
60,2022-01-01,S004,P0001,Toys,South,477,271,66,286.65,15.62,15,Sunny,0,16.09,Winter
80,2022-01-01,S005,P0001,Toys,West,111,77,60,83.47,70.33,10,Cloudy,0,70.29,Winter


P0002


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
21,2022-01-01,S002,P0002,Clothing,West,373,141,151,135.62,46.16,10,Rainy,1,45.21,Winter
41,2022-01-01,S003,P0002,Electronics,West,115,4,169,11.11,28.20,10,Sunny,1,25.63,Spring
61,2022-01-01,S004,P0002,Clothing,North,462,180,183,176.06,41.13,5,Sunny,0,45.05,Autumn
81,2022-01-01,S005,P0002,Clothing,West,415,298,107,292.96,57.15,5,Snowy,1,60.71,Autumn


## B1, B2 and B3 — Cell 5F: Selected preparation and comparison
Compare original daily data with parsed/sorted daily data and a separate monthly view. A complete target requires every calendar day and every daily sales value. Monthly lags use strictly preceding complete months; missing months remain missing. Full source attributes remain in the daily export. The candidate feature allowlist excludes future-unknown fields and the source Demand Forecast.

In [6]:
daily, monthly = prepare(frame)
print('Original:', frame.shape, 'Daily:', daily.shape, 'Monthly:', monthly.shape)
print('Complete months:', int(monthly.complete_month.sum()), 'Incomplete:', int((~monthly.complete_month).sum()))
display(daily.head())
display(monthly.loc[monthly['Store ID'].eq('S001') & monthly['Product ID'].eq('P0001')].tail(6))

Original: (73100, 15) Daily: (73100, 15) Monthly: (2500, 24)
Complete months: 2400 Incomplete: 100


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


,Store ID,Product ID,month,observed_days,nonmissing_sales_days,units_sold,inventory_last_observed,last_observed_date,category_count,region_count,...,lag_1,lag_2,lag_3,lag_12,rolling_mean_3,rolling_mean_6,sales_trend,history_ready,eligible_for_training,split
19,S001,P0001,2023-08-01,31,31,4766,154,2023-08-31,5,4,...,4527.0,3890.0,3388.0,4846.0,3935.000000,4135.666667,637.0,True,True,validation
20,S001,P0001,2023-09-01,30,30,3748,343,2023-09-30,5,4,...,4766.0,4527.0,3890.0,4592.0,4394.333333,4197.333333,239.0,True,True,validation
21,S001,P0001,2023-10-01,31,31,4256,433,2023-10-31,5,4,...,3748.0,4766.0,4527.0,4417.0,4347.000000,4039.833333,-1018.0,True,True,test
22,S001,P0001,2023-11-01,30,30,4131,417,2023-11-30,5,4,...,4256.0,3748.0,4766.0,3677.0,4256.666667,4095.833333,508.0,True,True,test
23,S001,P0001,2023-12-01,31,31,4120,135,2023-12-31,5,4,...,4131.0,4256.0,3748.0,4099.0,4045.000000,4219.666667,-125.0,True,True,test
24,S001,P0001,2024-01-01,1,1,40,223,2024-01-01,1,1,...,4120.0,4131.0,4256.0,3963.0,4169.000000,4258.000000,-11.0,True,False,ineligible


## B4 — Cells 6A and 6B: Export and verify lineage
Missing optional historical features are retained for Topic 4. Do not blindly use all exported columns as predictors.

In [7]:
part_b = run_pipeline(BASE_DIR, 'processed')
saved = json.loads((BASE_DIR / 'data/week3_manifest.json').read_text())
assert saved['records']
for record in saved['records']:
    assert sha256_file(source) == record['source_sha256']
    assert sha256_file(BASE_DIR / record['prepared_output']) == record['prepared_sha256']
print('COMMON LINEAGE VALIDATION: PASS')

SUCCESS: processed: 73100 daily rows; 2500 monthly rows
COMMON LINEAGE VALIDATION: PASS


## C1 — Cell 8F: Execute V1 batch pipeline
Expands the sales/inventory preprocessing component of Week 2. Batch ETL is appropriate for this static CSV. Schema/date errors stop before preparation; readback/hash mismatches fail validation. No model accuracy or full NFR benchmark is claimed.

In [8]:
first = run_pipeline(BASE_DIR)
print(json.dumps(first['inspection'], indent=2))

SUCCESS: pipeline_v1/timeseries: 73100 daily rows; 2500 monthly rows
{
  "rows": 73100,
  "columns": [
    "Date",
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Demand Forecast",
    "Price",
    "Discount",
    "Weather Condition",
    "Holiday/Promotion",
    "Competitor Pricing",
    "Seasonality"
  ],
  "dtypes": {
    "Date": "datetime64[ns]",
    "Store ID": "object",
    "Product ID": "object",
    "Category": "object",
    "Region": "object",
    "Inventory Level": "int64",
    "Units Sold": "int64",
    "Units Ordered": "int64",
    "Demand Forecast": "float64",
    "Price": "float64",
    "Discount": "int64",
    "Weather Condition": "object",
    "Holiday/Promotion": "int64",
    "Competitor Pricing": "float64",
    "Seasonality": "object"
  },
  "missing_by_column": {
    "Date": 0,
    "Store ID": 0,
    "Product ID": 0,
    "Category": 0,
    "Region": 0,
    "Inventory Level": 0,
    "Un

## Cell 9: Rerun and compare fingerprints, sizes and mappings

In [9]:
second = run_pipeline(BASE_DIR)
repeatability = verify_repeatability(first, second)
write_json(BASE_DIR / 'data/pipeline_v1/timeseries/repeatability_report.json', repeatability)
print(repeatability)
for left, right in zip(part_b['records'], first['records']):
    assert left['prepared_sha256'] == right['prepared_sha256']
print('Part B and Part C content parity: PASS')

SUCCESS: pipeline_v1/timeseries: 73100 daily rows; 2500 monthly rows
{'status': 'PASS', 'checks': ['source hashes and sizes', 'output hashes and sizes', 'counts', 'source-output mappings', 'feature allowlist'], 'scope': 'Same-runtime repeatability; execution timestamps intentionally excluded'}
Part B and Part C content parity: PASS


## C2–C4: Save artefacts
The delivered docs include the inventory, decisions and pipeline PNG plus editable Draw.io source. Source licence and upload equivalence remain unverified. January 2024 is partial; category and region identifiers vary. No GitHub publication has occurred. Append README_WEEK3_APPEND.md to the existing README when authorised repository access is available. Save this executed notebook separately. /content is temporary.

In [10]:
DOWNLOAD_EXPORTS = False
if DOWNLOAD_EXPORTS:
    import shutil
    from google.colab import files
    export_zip = shutil.make_archive(str(BASE_DIR / 'week3_data_exports'), 'zip', BASE_DIR / 'data')
    files.download(export_zip)